# 📈 Financial Analyst Agent over OpenBB Market Data

## Learning Objectives
In this notebook, you will learn:
1. **Market-data tools with OpenBB** - Ticker lookup, price metrics, fundamentals and market movers from providers that need no API key, plus news from web search
2. **Compact tool output** - Summarize a year of daily prices in code (returns, 52-week range) instead of handing the model 250 rows
3. **A scoped analyst agent** - One system prompt with two in-scope flows (market overview, specific company) and a polite refusal for everything else, checked on one request of each kind
4. **Follow-up questions** - A checkpointer lets "And Intel?" and "Which is better?" build on earlier answers; the numbers in the answers are checked against the tool data

## Where this fits
`01_research_assistant_agent.ipynb` answered from web pages. Here the agent answers from **structured market data**, where every number can be checked. The ReAct loop and `create_agent` come from Modules 03–04; this notebook is about the application: the tools, the scope rules and the checks.

## Prerequisites
- `create_agent` and checkpointers from Modules 03–04 and `01_research_assistant_agent.ipynb` (reading only)
- Dependencies installed with **`uv sync --group finance`** at the repository root (adds OpenBB; a later plain `uv sync` removes it again)
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`. No OpenBB key is needed: the tools use the SEC and Yahoo Finance providers, which are free and key-less

> ⚠️ This notebook is an engineering exercise, not investment advice. Market data changes every day, so your answers will differ from anyone else's.

---
## 📦 Step 1: Environment Setup

Load the API keys and import OpenBB (the first import builds OpenBB's provider catalogue and takes a few seconds).

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# OPENBB: the market-data SDK (installed by `uv sync --group finance`)
# ============================================================================
from openbb import obb

print(f"✅ OpenBB ready — providers used here: sec, yfinance "
      f"(installed: {len(obb.coverage.providers)} providers)")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the financial analyst
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: The Market-Data Tools

Five tools cover the questions an investor asks. Each one calls OpenBB (or Tavily for news) and returns **short markdown** the model can read.

| Tool | Data | Provider |
|---|---|---|
| `get_stock_ticker_symbol` | company name → ticker, SEC name, CIK | SEC (free) |
| `get_stock_price_metrics` | latest quote + 1-month … 1-year returns and 52-week range, computed from a year of daily prices | Yahoo Finance (free) |
| `get_stock_fundamental_indicator_metrics` | P/E, forward P/E, margins, growth, debt/equity, … | Yahoo Finance (free) |
| `get_stock_news` | recent headlines with dates and links | Tavily news search |
| `get_general_market_data` | most active stocks, top gainers, top losers | Yahoo Finance (free) |

### Key Concepts:
- **Key-less providers**: OpenBB routes each call to a *provider*. FMP, Intrinio or Polygon need their own API keys; SEC and Yahoo Finance do not, so the notebook runs with no extra setup
- **Summarize in code**: a year of daily prices is ~250 rows. The tool computes the few numbers an analyst reads (returns over 1/3/6/12 months, 52-week high/low) and returns those instead
- **Say what is missing**: a company with negative earnings has no trailing P/E. The tool says so explicitly, so the model cannot mistake a missing value for a zero or fill it in

In [ ]:
# ============================================================================
# MARKET-DATA TOOLS
# ============================================================================
from datetime import datetime, timedelta

import pandas as pd
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

news_search = TavilySearch(max_results=5, topic="news", time_range="month")


@tool
def get_stock_ticker_symbol(stock_name: str) -> str:
    """Get the ticker symbol, name and CIK for a publicly traded company from its name."""
    try:
        df = obb.equity.search(stock_name, provider="sec").to_df().head(5)
    except Exception:
        return (f"No company found for '{stock_name}'. "
                "Try correcting the spelling or a shorter version of the name.")
    return "Matching companies (best match first):\n\n" + df.to_markdown(index=False)


@tool
def get_stock_price_metrics(stock_ticker: str) -> str:
    """Get the latest price quote and price performance (1-month to 1-year returns,
    52-week range) for a stock ticker."""
    quote = obb.equity.price.quote(stock_ticker, provider="yfinance").to_df().iloc[0]
    start = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")
    close = obb.equity.price.historical(stock_ticker, start_date=start,
                                        provider="yfinance").to_df()["close"]
    last = close.iloc[-1]
    # Return over N trading days (≈21 per month), computed here — not by the model
    perf = {f"return_{label}": f"{(last / close.iloc[-days - 1] - 1):+.1%}"
            for label, days in [("1m", 21), ("3m", 63), ("6m", 126), ("1y", len(close) - 1)]
            if len(close) > days}
    summary = {
        "symbol": stock_ticker.upper(), "name": quote["name"], "currency": quote["currency"],
        "last_price": round(quote["last_price"], 2), "prev_close": round(quote["prev_close"], 2),
        "day_range": f"{quote['low']:.2f} – {quote['high']:.2f}",
        "52w_high": round(close.max(), 2), "52w_low": round(close.min(), 2),
        "ma_50d": round(quote["ma_50d"], 2), "ma_200d": round(quote["ma_200d"], 2), **perf,
    }
    return pd.Series(summary).to_markdown()


KEY_FUNDAMENTALS = ["pe_ratio", "forward_pe", "market_cap"]


@tool
def get_stock_fundamental_indicator_metrics(stock_ticker: str) -> str:
    """Get fundamental indicators for a stock ticker: P/E and forward P/E, market cap,
    margins, growth, returns on assets/equity, debt to equity, dividend yield and more."""
    metrics = obb.equity.fundamental.metrics(stock_ticker, provider="yfinance").to_df().iloc[0]
    metrics = metrics.dropna()
    missing = [k for k in KEY_FUNDAMENTALS if k not in metrics.index]
    note = ("\n\nNOT AVAILABLE from the provider: " + ", ".join(missing) +
            " (a missing pe_ratio usually means trailing earnings are negative)") if missing else ""
    return f"Fundamentals for {stock_ticker.upper()}:\n\n{metrics.to_markdown()}{note}"


@tool
def get_stock_news(stock_ticker: str) -> str:
    """Get recent news headlines (last month) for a stock ticker or company."""
    hits = news_search.invoke({"query": f"{stock_ticker} stock news"})["results"]
    lines = [f"- {h.get('published_date', '')[:16]} | {h['title']} | {h['url']}" for h in hits]
    return f"Recent news for {stock_ticker.upper()}:\n" + "\n".join(lines)


@tool
def get_general_market_data() -> str:
    """Get an overview of today's stock market: most actively traded stocks, top price
    gainers and top price losers. Useful for general ideas on which stocks to look at."""
    cols = ["symbol", "name", "price", "percent_change", "volume"]
    sections = []
    for title, fetch in [("Most active", obb.equity.discovery.active),
                         ("Top gainers", obb.equity.discovery.gainers),
                         ("Top losers", obb.equity.discovery.losers)]:
        df = fetch(provider="yfinance", limit=10).to_df()[cols]
        # The provider returns percent_change as a fraction (0.05 = +5%): format it for the model
        df["percent_change"] = df["percent_change"].map(lambda x: f"{x:+.2%}")
        sections.append(f"{title}:\n\n{df.to_markdown(index=False)}")
    return "\n\n".join(sections)


tools = [get_stock_ticker_symbol, get_stock_price_metrics,
         get_stock_fundamental_indicator_metrics, get_stock_news, get_general_market_data]
for t in tools:
    print(f"✅ {t.name}: {t.description.splitlines()[0]}")

In [ ]:
# ============================================================================
# TRY THE TOOLS DIRECTLY (no LLM)
# ============================================================================
print(get_stock_ticker_symbol.invoke("Nvidia"), "\n")
print(get_stock_ticker_symbol.invoke("microsfot"), "\n")
print(get_stock_price_metrics.invoke("NVDA"), "\n")
fundamentals_intc = get_stock_fundamental_indicator_metrics.invoke("INTC")
print(fundamentals_intc[:300], "...", fundamentals_intc[-160:], "\n")
print(get_stock_news.invoke("NVDA")[:600], "\n")
print(get_general_market_data.invoke({})[:700])

Two outputs to notice: the misspelled "microsfot" returns a hint instead of an error, and Intel's fundamentals end with an explicit **NOT AVAILABLE** line (no trailing P/E while earnings are negative).

---
## 🧭 Part 2: The Analyst Agent and Its Scope

The system prompt defines the analyst's job as two flows and a refusal:
- **Flow 1**: general market questions → market movers (and news if needed)
- **Flow 2**: a specific company → look up the ticker (fixing spelling if the lookup fails), then fetch only the data the question needs
- **Anything else** → a friendly "I can only help with market data"

The prompt is the one written for the original financial-analyst agents, with today's date added so "right now" and "recent" mean today.

In [ ]:
# ============================================================================
# SYSTEM PROMPT + AGENT
# ============================================================================
from datetime import date

from langchain.agents import create_agent

AGENT_PREFIX = """Role: You are an AI stock market assistant tasked with providing investors
with up-to-date, detailed information on individual stocks or advice based on general market data.

Objective: Assist data-driven stock market investors by giving accurate,
complete, but concise information relevant to their questions about individual
stocks or general advice on useful stocks based on general market data and trends.

Capabilities: You are given a number of tools as functions. Use as many tools
as needed to ensure all information provided is timely, accurate, concise,
relevant, and responsive to the user's query.

Starting Flow:
Input validation. Determine if the input is asking about a specific company
or stock ticker (Flow 2). If not, check if they are asking for general advice on potentially useful stocks
based on current market data (Flow 1). Otherwise, respond in a friendly, positive, professional tone
that you don't have information to answer as you can only provide financial advice based on market data.
For each of the flows related to valid questions use the following instructions:

Flow 1:
A. Market Analysis: If the query is valid and the user wants to get general advice on the market
or stocks worth looking into for investing, leverage the general market data tool to get relevant data.

Flow 2:
A. Symbol extraction. If the query is valid and is related to a specific company or companies,
extract the company name or ticker symbol from the question.
If a company name is given, look up the ticker symbol using a tool.
If the ticker symbol is not found based on the company, try to
correct the spelling and try again, like changing "microsfot" to "microsoft",
or broadening the search, like changing "southwest airlines" to a shorter variation
like "southwest". If the company or ticker is
still unclear based on the question or conversation so far, and the results of the
symbol lookup, then ask the user to clarify which company or ticker.

B. Information retrieval. Determine what data the user is seeking on the symbol
identified. Use the appropriate tools to fetch the requested information. Only use
data obtained from the tools. You may use multiple tools in a sequence. For instance,
first determine the company's symbol, then retrieve price data using the symbol
and fundamental indicator data etc. For specific queries only retrieve data using the most relevant tool.
If detailed analysis is needed, you can call multiple tools to retrieve data first.

Response Generation Flow:
Compose Response. Analyze the retrieved data carefully and provide a comprehensive answer to the user in a clear and concise format,
in a friendly professional tone, emphasizing the data retrieved.
If the user asks for recommendations you can give some recommendations
but emphasize the user to do their own research before investing.
When generating the final response in markdown,
if there are special characters in the text, such as the dollar symbol,
ensure they are escaped properly for correct rendering e.g $25.5 should become \\$25.5

Example Interaction:
User asks: "What is the PE ratio for Eli Lilly?"
Chatbot recognizes 'Eli Lilly' as a company name.
Chatbot uses symbol lookup to find the ticker for Eli Lilly, returning LLY.
Chatbot retrieves the PE ratio using the proper function with symbol LLY.
Chatbot responds: "The PE ratio for Eli Lilly (symbol: LLY) as of May 12, 2024 is 30."

Check carefully and only call the tools which are specifically named below.
Only use data obtained from these tools.
"""

SYS_PROMPT = f"Today's date is {date.today():%B %d, %Y}.\n\n" + AGENT_PREFIX

financial_analyst = create_agent(llm, tools, system_prompt=SYS_PROMPT)
print(f"✅ Agent graph nodes: {list(financial_analyst.get_graph().nodes)}")

In [ ]:
# ============================================================================
# ONE REQUEST PER FLOW: check which tools the agent used
# ============================================================================
from IPython.display import Markdown, display


def tool_calls(result) -> list[tuple[str, dict]]:
    return [(tc["name"], tc["args"]) for m in result["messages"]
            for tc in getattr(m, "tool_calls", [])]


scenarios = [
    ("Flow 1 — market overview", "Tell me some stocks I could potentially look into today.",
     lambda calls: any(n == "get_general_market_data" for n, _ in calls)),
    ("Flow 2 — misspelled company", "What is the P/E ratio of microsfot?",
     lambda calls: any(n == "get_stock_fundamental_indicator_metrics"
                       and a.get("stock_ticker", "").upper() == "MSFT" for n, a in calls)),
    ("Out of scope", "Can you recommend a good pasta recipe for tonight?",
     lambda calls: not calls),
]

results = {}
for label, question, passes in scenarios:
    result = financial_analyst.invoke({"messages": [{"role": "user", "content": question}]})
    calls = tool_calls(result)
    results[label] = result
    print(f"{'✅' if passes(calls) else '❌'} {label}: {question}")
    for name, args in calls:
        print(f"      → {name}({args})")
    print(f"   Answer: {result['messages'][-1].content[:300]}...\n")

The misspelled request shows the prompt's recovery rule at work: the first lookup fails, the agent retries with the corrected name, then fetches the fundamentals for `MSFT`. The out-of-scope request uses no tools at all.

---
## 💬 Part 3: Follow-Up Questions

Investors rarely ask one isolated question. With a checkpointer, the agent keeps the conversation, so "And Intel?" and "Which is better?" refer back to earlier answers (the mechanism is the same as in `01_research_assistant_agent.ipynb`).

In [ ]:
# ============================================================================
# A THREE-TURN CONVERSATION
# ============================================================================
from langgraph.checkpoint.memory import InMemorySaver

chat_analyst = create_agent(llm, tools, system_prompt=SYS_PROMPT, checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "investor-1"}}

answers = {}
for turn in ["What is the P/E ratio of Nvidia?",
             "And Intel?",
             "Which of the two looks better on valuation? Keep it short."]:
    out = chat_analyst.invoke({"messages": [{"role": "user", "content": turn}]}, thread)
    answers[turn] = out["messages"][-1].content
    print(f"👤 {turn}")
    display(Markdown(answers[turn]))

### 3.1 ✅ Checking the numbers

The P/E values in the answers must match what Yahoo Finance returns right now. The check fetches the same data directly and compares:
- **Nvidia**: its trailing P/E must appear in the first answer (rounded to 1 or 2 decimals)
- **Intel**: there is no trailing P/E while its earnings are negative, so the answer must either say so or quote the forward P/E — any other P/E figure would be invented

In [ ]:
# ============================================================================
# CHECK THE ANSWERS AGAINST THE PROVIDER DATA
# ============================================================================
import re


def pe_values(ticker: str) -> dict:
    m = obb.equity.fundamental.metrics(ticker, provider="yfinance").to_df().iloc[0]
    return {k: (None if pd.isna(m.get(k)) else float(m.get(k))) for k in ("pe_ratio", "forward_pe")}


def mentions(value: float, text: str) -> bool:
    """True if the answer contains the value rounded to 1 or 2 decimals (or truncated)."""
    numbers = {float(x) for x in re.findall(r"\d+\.\d+|\d+", text.replace(",", ""))}
    return any(abs(n - value) < 0.051 for n in numbers)


nvda, intc = pe_values("NVDA"), pe_values("INTC")
print(f"Provider data  NVDA: {nvda}\n               INTC: {intc}\n")

a_nvda, a_intc = answers["What is the P/E ratio of Nvidia?"], answers["And Intel?"]
print(("✅" if nvda["pe_ratio"] and mentions(nvda["pe_ratio"], a_nvda) else "❌"),
      "Nvidia answer quotes the provider's trailing P/E")

says_unavailable = any(k in a_intc.lower() for k in
                       ["not available", "n/a", "negative", "unavailable", "no trailing",
                        "not applicable", "does not have", "doesn't have", "isn't available"])
quotes_forward = intc["forward_pe"] is not None and mentions(intc["forward_pe"], a_intc)
if intc["pe_ratio"] is not None:
    ok = mentions(intc["pe_ratio"], a_intc)
else:
    ok = says_unavailable or quotes_forward
print(("✅" if ok else "❌"), "Intel answer is consistent with the provider data "
      f"(says unavailable: {says_unavailable}, quotes forward P/E: {quotes_forward})")

---
## 📝 Summary

In this notebook, we learned:

### 1. Market-data tools with OpenBB
- OpenBB puts many data providers behind one API; the SEC and Yahoo Finance providers need no key
- News comes from a web-search tool when the market-data provider has none

### 2. Compact tool output
- Compute returns and ranges in code and hand the model a short table; the model reads numbers, it should not do the arithmetic
- Make missing values explicit ("NOT AVAILABLE: pe_ratio") so a gap cannot turn into an invented figure

### 3. A scoped analyst agent
- The system prompt defines what is in scope and how to recover (spelling fixes); one request per flow shows each path is taken — including the refusal with no tool calls

### 4. Follow-up questions
- A checkpointer keeps the conversation, so comparative follow-ups work
- Numbers in the answers are checked against the same provider data, fetched directly

### Next Steps
- **`04_text2sql_workflow.ipynb`**: answering business questions from a SQL database with a fixed, deterministic chain